# Zero-shot LLM baseline (step 9) on Google Colab

Runs the scam-call classifier's **zero-shot LLM** (Ollama + `qwen2.5:3b`, no training) on a free Colab T4 GPU.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Cell 2 asks you to upload **`colab_bundle.zip`** (code + processed phone-call data, a few MB).
The last cell downloads **`llm_results.zip`** — send it back so the numbers go into the README.

Phone calls only; all data comes from the public datasets listed in `CLAUDE.md`.

In [ ]:
# 1. Check the GPU (should say Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload colab_bundle.zip and unpack it
import os, zipfile
from google.colab import files

if not os.path.exists('colab_bundle.zip'):
    files.upload()  # pick colab_bundle.zip
with zipfile.ZipFile('colab_bundle.zip') as z:
    z.extractall('scam-classifier')
%cd /content/scam-classifier
!ls src data/processed data/processed/splits | head -40

In [ ]:
# 3. Python packages (Colab already has pandas, numpy, scikit-learn, tqdm)
!pip -q install tabulate

In [ ]:
# 4. Install Ollama and start its server in the background, answering 8 calls at once
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import os, subprocess, time, urllib.request
PARALLEL = 8
env = dict(os.environ, OLLAMA_NUM_PARALLEL=str(PARALLEL), OLLAMA_KEEP_ALIVE='2h')
server = subprocess.Popen(['ollama', 'serve'], env=env,
                          stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print('Ollama up:', urllib.request.urlopen('http://localhost:11434/api/version').read().decode()); break
    except Exception:
        time.sleep(1)

In [ ]:
# 5. Download the model (~1.9 GB, about a minute on Colab)
MODEL = 'qwen2.5:3b'   # optional later: 'qwen2.5:7b' fits on a T4 too
!ollama pull {MODEL}
!ollama run {MODEL} "Reply with exactly: OK"

In [ ]:
# 6. Run the zero-shot LLM experiment
#    SAMPLE calls per test set (+ EARLY for the first-50-words test). 0 = use every call.
SAMPLE = 200
EARLY = 60
!python src/run_experiment.py --use-llm --llm-model {MODEL} --llm-sample {SAMPLE if SAMPLE else 100000} \
    --llm-early-sample {EARLY if EARLY else 100000} --llm-workers {PARALLEL}

In [ ]:
# 7. Show the results
from IPython.display import Markdown, display
display(Markdown(open('results/summary_llm.md').read()))

In [ ]:
# 8. Download results + the LLM answer cache (so nothing has to be re-run)
!zip -q -r llm_results.zip results/metrics_llm*.csv results/summary_llm.md data/processed/llm_cache_*.jsonl
files.download('llm_results.zip')